In [10]:
import_timit = True
import_lbs = False
import_sandi = False
import_ume_erj =  False

import_features = False
import_alignments = False

In [11]:
import inspect
import os
import subprocess
import sys
import time

from pathlib import Path
from tqdm import tqdm


from lhotse import (
    CutSet,
    load_manifest_lazy
)

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    Wav2Vec2Processor, 
    Wav2Vec2ForCTC 
)

#if you need to mount your code entry point
sys.path.append("native-L2-coordinate-geometry/src")


from config import (
    DATA_PATH,
    MANIFEST_PATH,
    SANDI_CEFR_DICT,
    MODELS,
)

from build_phone_codes import (
    AveragingStrategy,
    build_average_phone_matrix_for_timit,
    build_average_phone_matrix_from_tuple_alignments,
    build_phone_averages_from_pickles,
    
    
)

from utils import (
    get_cuts,
    load_lexicon,
    load_pickle,
    PhoneClass,
    save_pickle,
)

#############
#Variables
#############
#label for phone assessment
PHONE_CLASS = PhoneClass.DIPHONE
AVG_STRATEGY = AveragingStrategy.CENTER_PHONE

DEVICE='cuda'
FRAME_SHIFT = 0.02 
RANK = 16


#############
#Data
#############
#examples: either call get_cuts or use lhotse from_file

DATA_PARTS_TIMIT = ['TRAIN', 'DEV', 'TEST']
cutsets_timit = get_cuts(corpus_name='timit', partition_names=DATA_PARTS_TIMIT, manifest_dir=MANIFEST_PATH)
cuts_train_timit = cutsets_timit['TRAIN']
cuts_dev_timit = cutsets_timit['DEV']
cuts_test_timit = cutsets_timit['TEST']


sandi_cutset_path =  /path/to/sandi
cuts_train_sandi = CutSet.from_file(
    sandi_cutset_path
)



In [8]:
#models = MODELS

models = {'wavlm_base_plus': MODELS['wavlm_base_plus']}
models['wavlm_base_plus'] ['layer'] = [6]

In [ ]:
data_indices, task = (("train", cut_indices_sandi, cuts_train_sandi), ("dev", None, cuts_dev_sandi)), "sandi"

for name, info in models.items():
    print(f"Processing {name}")
    layers = [info["layer"]] if isinstance(info["layer"], int) else info["layer"]

    jobs = [
        (d, ind, c, layer, Path(f"{DATA_PATH}/features/{name}/{task}/layer{layer}/{d}"))
        for d, ind, c in data_indices
        for layer in layers
        if not any(Path(f"{DATA_PATH}/features/{name}/{task}/layer{layer}/{d}").glob("*.pickle"))
    ]

    if not jobs:
        print(f"All features found for {name}, skipping model.")
        continue

    model = None
    if info["hf_name"] is not None:
        model = AutoModel.from_pretrained(info["hf_name"]).to(DEVICE).eval()

    for d, ind, c, layer, output_dir in jobs:
        ensure_dir(output_dir)

        if model is not None:
            extract_and_save_features(
                cuts=c, model=model, split=d, output_dir=output_dir,
                batch_size=1000, device=DEVICE, layer=layer, indices=ind,
            )
        else:
            extract_and_save_mfcc(
                cuts=c, split=d, output_dir=output_dir,
                batch_size=1000, indices=ind, subsampling=2,
            )

    if model is not None:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    